In [2]:
import re
import string
import numpy as np
import pandas as pd
from collections import Counter
from heapq import nlargest
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    print("Please install spaCy model: python -m spacy download en_core_web_sm")

Please install spaCy model: python -m spacy download en_core_web_sm


In [3]:
class LegalDocumentPreprocessor:
    def __init__(self):
        self.legal_stopwords = {
            'hereinafter', 'whereas', 'therefore', 'herein', 'hereto', 'thereof',
            'aforementioned', 'aforesaid', 'notwithstanding', 'pursuant', 'insofar',
            'thereto', 'wheresoever', 'whatsoever', 'howsoever', 'whomsoever'
        }

    def clean_legal_text(self, text):
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()
        text = re.sub(r'\b\d+\.\s*', '', text)
        text = re.sub(r'\([A-Za-z]+\)', '', text)
        return text

    def extract_legal_entities(self, text):
        doc = nlp(text)

        entities = {
            'PERSON': [ent.text for ent in doc.ents if ent.label_ == 'PERSON'],
            'ORG': [ent.text for ent in doc.ents if ent.label_ == 'ORG'],
            'GPE': [ent.text for ent in doc.ents if ent.label_ == 'GPE'],
            'DATE': [ent.text for ent in doc.ents if ent.label_ == 'DATE'],
            'MONEY': [ent.text for ent in doc.ents if ent.label_ == 'MONEY']
        }

        return entities

    def tokenize_legal_text(self, text):
        doc = nlp(text)

        tokens = []
        for token in doc:
            if (not token.is_stop and
                not token.is_punct and
                not token.is_space and
                token.text.lower() not in self.legal_stopwords and
                len(token.text) > 2):
                tokens.append(token.lemma_.lower())

        return tokens

In [4]:
class LegalExtractiveSummarizer:
    def __init__(self, preprocessor):
        self.preprocessor = preprocessor
        self.tfidf_vectorizer = TfidfVectorizer(
            max_features=1000,
            stop_words='english',
            ngram_range=(1, 2)
        )

    def summarize(self, text, num_sentences=3):
        clean_text = self.preprocessor.clean_legal_text(text)
        doc = nlp(clean_text)

        sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 20]

        if len(sentences) <= num_sentences:
            return clean_text

        try:
            tfidf_matrix = self.tfidf_vectorizer.fit_transform(sentences)
            sentence_scores = np.array(tfidf_matrix.sum(axis=1)).flatten()
        except:
            sentence_scores = self._fallback_scoring(sentences)

        top_indices = nlargest(num_sentences, range(len(sentence_scores)), key=lambda i: sentence_scores[i])
        top_indices.sort()

        summary = ' '.join([sentences[i] for i in top_indices])
        return summary

    def _fallback_scoring(self, sentences):
        all_tokens = []
        for sent in sentences:
            tokens = self.preprocessor.tokenize_legal_text(sent)
            all_tokens.extend(tokens)

        word_freq = Counter(all_tokens)
        max_freq = max(word_freq.values()) if word_freq else 1

        for word in word_freq:
            word_freq[word] = word_freq[word] / max_freq

        scores = []
        for sent in sentences:
            tokens = self.preprocessor.tokenize_legal_text(sent)
            score = sum(word_freq.get(token, 0) for token in tokens)
            scores.append(score / len(tokens) if tokens else 0)

        return np.array(scores)

In [5]:
class LegalAbstractiveSummarizer:
    def __init__(self, model_name="google/flan-t5-large"):
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)
            self.model.eval()
            print(f"Loaded model: {self.model_name}")
        except Exception as e:
            print(f"Error loading model: {e}")

    def summarize(self, text, max_length=150, min_length=50):
        if self.model is None or self.tokenizer is None:
            return "Model not loaded properly"

        prompt = f"""Summarize this legal document in a clear and concise manner,
        focusing on the key obligations, rights, and terms:

{text}"""

        try:
            inputs = self.tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=1024
            )

            with torch.no_grad():
                outputs = self.model.generate(
                    inputs["input_ids"],
                    max_length=max_length,
                    min_length=min_length,
                    num_beams=4,
                    early_stopping=True,
                    temperature=0.7,
                    no_repeat_ngram_size=2
                )

            summary = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            return summary

        except Exception as e:
            return f"Error: {e}"

In [6]:
class DocumentReader:
    def __init__(self):
        self.supported_formats = ['.pdf', '.txt', '.md', '.docx']

    def read_document(self, file_path):
        try:
            file_path = str(file_path).strip()
            file_ext = Path(file_path).suffix.lower()

            if file_ext not in self.supported_formats:
                return {
                    'content': "",
                    'file_path': file_path,
                    'file_type': file_ext,
                    'success': False,
                    'error': f'Unsupported file format: {file_ext}. Supported: {self.supported_formats}'
                }

            if file_ext == '.pdf':
                return self._read_pdf(file_path)
            elif file_ext in ['.txt', '.md']:
                return self._read_text(file_path)
            elif file_ext == '.docx':
                return self._read_docx(file_path)

        except Exception as e:
            return {
                'content': "",
                'file_path': file_path,
                'file_type': 'unknown',
                'success': False,
                'error': f'Error reading file: {str(e)}'
            }

    def _read_pdf(self, file_path):
        content = ""
        method_used = ""

        try:
            from pdfminer.high_level import extract_text
            with open(file_path, 'rb') as file:
                content = extract_text(file)
            method_used = "pdfminer"
        except ImportError:
            pass
        except Exception as e:
            pass

        if not content.strip():
            try:
                import PyPDF2
                with open(file_path, 'rb') as file:
                    reader = PyPDF2.PdfReader(file)
                    content = ""
                    for page_num, page in enumerate(reader.pages):
                        page_text = page.extract_text()
                        if page_text.strip():
                            content += page_text + "\n"
                method_used = "PyPDF2"
            except ImportError:
                pass
            except Exception as e:
                pass

        if not content.strip():
            try:
                import pypdf
                with open(file_path, 'rb') as file:
                    reader = pypdf.PdfReader(file)
                    content = ""
                    for page_num, page in enumerate(reader.pages):
                        page_text = page.extract_text()
                        if page_text.strip():
                            content += page_text + "\n"
                method_used = "pypdf"
            except ImportError:
                pass
            except Exception as e:
                pass

        if not content.strip():
            try:
                import pdfplumber
                with pdfplumber.open(file_path) as pdf:
                    content = ""
                    for page_num, page in enumerate(pdf.pages):
                        page_text = page.extract_text()
                        if page_text.strip():
                            content += page_text + "\n"
                method_used = "pdfplumber"
            except ImportError:
                pass
            except Exception as e:
                pass

        if content.strip():
            word_count = len(content.split())
            char_count = len(content)

            return {
                'content': content,
                'file_path': file_path,
                'file_type': 'pdf',
                'success': True,
                'method_used': method_used,
                'word_count': word_count,
                'char_count': char_count,
                'pages_processed': len(content.split('\n\n')) if content else 0
            }
        else:
            return {
                'content': "",
                'file_path': file_path,
                'file_type': 'pdf',
                'success': False,
                'error': 'No text could be extracted. The PDF might be:\n- Scanned images (needs OCR)\n- Password-protected\n- Corrupted or empty'
            }

    def _read_text(self, file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                content = file.read()

            word_count = len(content.split())
            char_count = len(content)

            return {
                'content': content,
                'file_path': file_path,
                'file_type': 'text',
                'success': True,
                'word_count': word_count,
                'char_count': char_count
            }
        except UnicodeDecodeError:
            try:
                with open(file_path, 'r', encoding='latin-1') as file:
                    content = file.read()
                return {
                    'content': content,
                    'file_path': file_path,
                    'file_type': 'text',
                    'success': True,
                    'word_count': len(content.split()),
                    'encoding_used': 'latin-1'
                }
            except Exception as e:
                return {
                    'content': "",
                    'file_path': file_path,
                    'file_type': 'text',
                    'success': False,
                    'error': f'Encoding error: {str(e)}'
                }

    def _read_docx(self, file_path):
        try:
            from docx import Document
            doc = Document(file_path)
            content = ""

            for paragraph in doc.paragraphs:
                content += paragraph.text + "\n"

            word_count = len(content.split())
            char_count = len(content)

            return {
                'content': content,
                'file_path': file_path,
                'file_type': 'docx',
                'success': True,
                'word_count': word_count,
                'char_count': char_count,
                'paragraphs': len(doc.paragraphs)
            }
        except ImportError:
            return {
                'content': "",
                'file_path': file_path,
                'file_type': 'docx',
                'success': False,
                'error': 'python-docx not installed. Run: pip install python-docx'
            }
        except Exception as e:
            return {
                'content': "",
                'file_path': file_path,
                'file_type': 'docx',
                'success': False,
                'error': f'Error reading DOCX: {str(e)}'
            }

    def get_document_info(self, file_path):
        try:
            path = Path(file_path)
            if path.exists():
                stat = path.stat()
                return {
                    'file_exists': True,
                    'file_size': stat.st_size,
                    'file_size_mb': round(stat.st_size / (1024 * 1024), 2),
                    'modified_time': datetime.fromtimestamp(stat.st_mtime),
                    'file_extension': path.suffix.lower()
                }
            else:
                return {
                    'file_exists': False,
                    'error': f'File not found: {file_path}'
                }
        except Exception as e:
            return {
                'file_exists': False,
                'error': f'Error checking file: {str(e)}'
            }

In [7]:
class HybridLegalSummarizer:
    def __init__(self):
        self.preprocessor = LegalDocumentPreprocessor()
        self.extractive = LegalExtractiveSummarizer(self.preprocessor)
        self.abstractive = LegalAbstractiveSummarizer()

    def summarize(self, text, method='hybrid', **kwargs):
        if method == 'extractive':
            return self.extractive.summarize(text, **kwargs)

        elif method == 'abstractive':
            return self.abstractive.summarize(text, **kwargs)

        elif method == 'hybrid':
            extractive_summary = self.extractive.summarize(
                text,
                num_sentences=kwargs.get('num_sentences', 5)
            )

            final_summary = self.abstractive.summarize(
                extractive_summary,
                max_length=kwargs.get('max_length', 120),
                min_length=kwargs.get('min_length', 40)
            )

            return final_summary

        else:
            raise ValueError("Method must be 'extractive', 'abstractive', or 'hybrid'")

    def get_summary_with_metadata(self, text, method='hybrid'):
        entities = self.preprocessor.extract_legal_entities(text)
        summary = self.summarize(text, method)

        original_length = len(text.split())
        summary_length = len(summary.split())
        compression_ratio = summary_length / original_length if original_length > 0 else 0

        metadata = {
            'method': method,
            'original_length': original_length,
            'summary_length': summary_length,
            'compression_ratio': round(compression_ratio, 3),
            'entities': entities,
            'summary': summary
        }

        return metadata

In [8]:
!pip install pdfminer.six PyPDF2 pypdf pdfplumber python-docx -q
!pip install pytesseract pdf2image -q
!pip install transformers torch sentencepiece -q

try:
    import pdfminer
    print("pdfminer.six installed")
except ImportError:
    print("pdfminer.six failed to install")

try:
    import PyPDF2
    print("PyPDF2 installed")
except ImportError:
    print("PyPDF2 failed to install")

try:
    import pypdf
    print("pypdf installed")
except ImportError:
    print("pypdf failed to install")

try:
    import pdfplumber
    print("pdfplumber installed")
except ImportError:
    print("pdfplumber failed to install")

^C


pdfminer.six installed
PyPDF2 installed
pypdf installed
pdfplumber installed


In [ ]:
import os
from pathlib import Path

document_path = '/content/casediary.pdf'

if os.path.exists(document_path):
    print(f"File found: {document_path}")
    file_size = os.path.getsize(document_path) / (1024 * 1024)
    print(f"File size: {file_size:.2f} MB")
else:
    print(f"File not found: {document_path}")
    legal_text = """
THIS SERVICE AGREEMENT ("Agreement") is made and entered into on this 15th day of January, 2026,
by and between AlphaTech Solutions Pvt. Ltd., a company incorporated under the Companies Act,
2013, having its registered office at Bengaluru, Karnataka (hereinafter referred to as the
"Service Provider"), and Orion Financial Services Ltd., having its principal place of business
at Mumbai, Maharashtra (hereinafter referred to as the "Client").

WHEREAS, the Client desires to engage the Service Provider for the purpose of developing,
implementing, and maintaining a customized artificial intelligence-based document management
system; and

WHEREAS, the Service Provider represents that it possesses the requisite expertise, personnel,
and technical capabilities to perform the services described herein;

NOW, THEREFORE, in consideration of the mutual covenants and promises contained herein,
the parties agree as follows:

1. SCOPE OF SERVICES
The Service Provider shall design, develop, test, deploy, and maintain an AI-powered
document classification and summarization platform capable of processing financial and
legal documents. The platform shall include natural language processing modules, secure
cloud hosting infrastructure, and data encryption protocols compliant with applicable laws.

2. TERM AND TERMINATION
This Agreement shall commence on the Effective Date and shall remain in force for a period
of three (3) years unless terminated earlier in accordance with this clause. Either party may
terminate this Agreement upon ninety (90) days' prior written notice. Immediate termination
may occur in the event of material breach, insolvency, or violation of confidentiality obligations.

3. CONFIDENTIALITY
Both parties acknowledge that during the term of this Agreement, they may have access to
confidential information including trade secrets, proprietary algorithms, financial data,
customer information, and strategic plans. Each party agrees to maintain strict confidentiality
and shall not disclose such information to any third party without prior written consent,
except as required by law.

4. INTELLECTUAL PROPERTY
All intellectual property developed specifically for the Client under this Agreement shall
remain the exclusive property of the Client. Pre-existing intellectual property of the Service
Provider shall remain its sole property. Any jointly developed intellectual property shall be
governed by a separate written agreement.

5. PAYMENT TERMS
The Client agrees to pay the Service Provider a total contract value of INR 2,50,00,000
(Two Crores Fifty Lakhs Only), payable in milestone-based installments. Delayed payments
shall attract interest at the rate of 12% per annum.

6. LIMITATION OF LIABILITY
Neither party shall be liable for indirect, incidental, or consequential damages, including
loss of profits or data. The total aggregate liability under this Agreement shall not exceed
the total fees paid under this Agreement.

7. GOVERNING LAW AND JURISDICTION
This Agreement shall be governed by and construed in accordance with the laws of India.
The courts of Bengaluru shall have exclusive jurisdiction over any disputes arising
under this Agreement.

IN WITNESS WHEREOF, the parties hereto have executed this Agreement as of the date first written above.
"""
    print("Using sample legal document for demonstration")

if 'legal_text' not in locals() or not legal_text.strip():
    print(f"Attempting to read PDF: {document_path}")

    reader = DocumentReader()

    try:
        doc_data = reader.read_document(document_path)

        if doc_data['success'] and doc_data['content'].strip():
            legal_text = doc_data['content']
            word_count = len(legal_text.split())
            print(f"PDF successfully read!")
            print(f"Word count: {word_count:,}")
            print(f"Method used: {doc_data.get('method_used', 'unknown')}")

            preview = legal_text[:400].replace('\n', ' ')
            print(f"Content preview:")
            print(f"   {preview}...")

        else:
            print(f"Failed to read PDF: {doc_data.get('error', 'Unknown error')}")

    except Exception as e:
        print(f"Error reading PDF: {e}")

print(f"Final document length: {len(legal_text.split())} words")

❌ File not found: /content/casediary.pdf
💡 Please upload your PDF file to Colab first!
📝 Using sample legal document for demonstration

📊 Final document length: 470 words


In [ ]:
preprocessor = LegalDocumentPreprocessor()
summarizer = HybridLegalSummarizer()

print("\nGENERATING SUMMARIES USING MULTIPLE METHODS")
print("-" * 50)

print("\n1️⃣ EXTRACTIVE SUMMARY (Sentence Selection):")
extractive_summary = summarizer.extractive.summarize(legal_text, num_sentences=8)
print(f"   {extractive_summary}")
print(f"   Length: {len(extractive_summary.split())} words")

print("\n2️⃣ ABSTRACTIVE SUMMARY (AI-Generated):")
abstractive_summary = summarizer.abstractive.summarize(
    legal_text,
    max_length=200,
    min_length=100
)
print(f"   {abstractive_summary}")
print(f"   Length: {len(abstractive_summary.split())} words")

print("\n3️⃣ HYBRID SUMMARY (Recommended):")
hybrid_summary = summarizer.summarize(
    legal_text,
    method='hybrid',
    num_sentences=10,
    max_length=250,
    min_length=120
)
print(f"   {hybrid_summary}")
print(f"   Length: {len(hybrid_summary.split())} words")

print("\nCOMPREHENSIVE DOCUMENT ANALYSIS")
print("-" * 50)

analysis_result = summarizer.get_summary_with_metadata(legal_text, method='hybrid')

original_length = analysis_result['original_length']
summary_length = analysis_result['summary_length']
compression_ratio = analysis_result['compression_ratio']

print(f"\nDOCUMENT STATISTICS:")
print(f"   Original Document: {original_length:,} words")
print(f"   Summary Generated: {summary_length:,} words")
print(f"   Compression Ratio: {compression_ratio:.3f}")
print(f"   Reduction: {(1-compression_ratio)*100:.1f}%")

risk_words = ["fraud", "penalty", "violation", "risk", "lawsuit", "breach",
              "noncompliance", "litigation", "regulatory", "fine", "termination",
              "liability", "confidential", "proprietary"]

text_lower = legal_text.lower()
detected_risks = [word for word in risk_words if word in text_lower]

print(f"\nRISK ASSESSMENT:")
if detected_risks:
    risk_categories = {
        'HIGH': ['fraud', 'lawsuit', 'breach'],
        'MEDIUM': ['penalty', 'violation', 'noncompliance', 'termination', 'liability'],
        'LOW': ['risk', 'litigation', 'regulatory', 'fine', 'confidential', 'proprietary']
    }

    for category, risks in risk_categories.items():
        category_risks = [risk for risk in detected_risks if risk in risks]
        if category_risks:
            print(f"   {category}: {', '.join(category_risks)}")
else:
    print("   No significant legal risks detected")

compliance_issues = []
if "governing law" not in text_lower and "jurisdiction" not in text_lower:
    compliance_issues.append("Missing governing law or jurisdiction clause")
if "termination" not in text_lower:
    compliance_issues.append("No termination clause found")
if "confidential" not in text_lower and "proprietary" not in text_lower:
    compliance_issues.append("No confidentiality clause")
if "liability" not in text_lower:
    compliance_issues.append("Liability terms not clearly defined")

print(f"\nCOMPLIANCE ANALYSIS:")
if compliance_issues:
    for issue in compliance_issues:
        print(f"   {issue}")
else:
    print("   Document appears compliant with standard legal requirements")

entities = analysis_result['entities']
print(f"\nLEGAL ENTITY RECOGNITION:")
if entities:
    for entity_type, entity_list in entities.items():
        if entity_list:
            print(f"   {entity_type}: {', '.join(entity_list[:5])}")
            if len(entity_list) > 5:
                print(f"      ... and {len(entity_list)-5} more")
else:
    print("   No specific legal entities identified")

print(f"\nCOMPREHENSIVE FINAL SUMMARY (10+ Lines)")
print("=" * 50)

final_summary = hybrid_summary

final_summary = '\n'.join(line.strip() for line in final_summary.split('\n') if line.strip())
print(final_summary)

print(f"\nSUMMARY STATISTICS:")
final_word_count = len(final_summary.split())
final_line_count = len([line for line in final_summary.split('\n') if line.strip()])
print(f"   Total Words: {final_word_count}")
print(f"   Total Lines: {final_line_count}")
print(f"   Requirements Met: {'YES' if final_word_count >= 500 or final_line_count >= 10 else 'NO'}")

summary_result = analysis_result
summary_result['comprehensive_summary'] = final_summary
summary_result['extractive_summary'] = extractive_summary
summary_result['abstractive_summary'] = abstractive_summary
summary_result['hybrid_summary'] = hybrid_summary
summary_result['detected_risks'] = detected_risks
summary_result['compliance_issues'] = compliance_issues

print(f"\nProcessing completed successfully!")

In [ ]:
document_path = '/content/bankj.PDF'

preprocessor = LegalDocumentPreprocessor()
summarizer = HybridLegalSummarizer()
reader = DocumentReader()

print(f"Reading document from: {document_path}")

try:
    doc_data = reader.read_document(document_path)
    legal_text = doc_data['content']

    print(f"Document content loaded. Original length: {len(legal_text.split())} words")
except Exception as e:
    print(f"Error reading document: {e}")
if legal_text:
    print("\nGenerating summary using 'hybrid' method...")
    summary_result = summarizer.get_summary_with_metadata(legal_text, method='hybrid')

    risk_words = ["fraud", "penalty", "violation", "risk", "lawsuit", "breach", "noncompliance", "litigation", "regulatory", "fine"]
    text_lower = legal_text.lower()
    risks = [word for word in risk_words if word in text_lower]

    compliance_issues = []
    if "governing law" not in text_lower and "jurisdiction" not in text_lower:
        compliance_issues.append("Missing governing law or jurisdiction clause")
    if "termination" not in text_lower:
        compliance_issues.append("No termination clause found")
    if "confidential" not in text_lower and "proprietary" not in text_lower:
        compliance_issues.append("No confidentiality clause")

    print("Processing completed")
else:
    print("No document content to process")

if legal_text:
    print("\n--- Summarization Result ---")
    print(f"Original Length: {summary_result['original_length']} words")
    print(f"Summary Length: {summary_result['summary_length']} words")
    print(f"Compression Ratio: {summary_result['compression_ratio']:.3f}")

    print("\nSummary:")
    print(summary_result['summary'])

    print("\nExtracted Legal Entities:")
    for entity_type, entity_list in summary_result['entities'].items():
        if entity_list:
            print(f"   - {entity_type}: {', '.join(entity_list)}")

    print("\nRisk Assessment:")
    if risks:
        for risk in risks:
            print(f"   - {risk.title()}")
    else:
        print("   - No significant risks detected")

    print("\nCompliance Check:")
    if compliance_issues:
        for issue in compliance_issues:
            print(f"   - {issue}")
    else:
        print("   - Document appears compliant")
else:
    print("No results to display")

if legal_text:
    print("\n--- Summarization Result ---")
    print(f"Original Length: {summary_result['original_length']} words")
    print(f"Summary Length: {summary_result['summary_length']} words")
    print(f"Compression Ratio: {summary_result['compression_ratio']:.3f}")

    print("\nSummary:")
    print(summary_result['summary'])

    print("\nExtracted Legal Entities:")
    for entity_type, entity_list in summary_result['entities'].items():
        if entity_list:
            print(f"   - {entity_type}: {', '.join(entity_list)}")

    print("\nRisk Assessment:")
    if risks:
        for risk in risks:
            print(f"   - {risk.title()}")
    else:
        print("   - No significant risks detected")

    print("\nCompliance Check:")
    if compliance_issues:
        for issue in compliance_issues:
            print(f"   - {issue}")
    else:
        print("   - Document appears compliant")
else:
    print("No results to display")
if legal_text:
    print("\n--- Method Comparison ---")

    methods = ['extractive', 'abstractive', 'hybrid']
    for method in methods:
        print(f"\n{method.upper()} METHOD:")
        result = summarizer.get_summary_with_metadata(legal_text, method)
        print(f"   Summary: {result['summary']}")
        print(f"   Compression: {result['compression_ratio']:.3f}")
if legal_text:
    print("\n" + "="*60)
    print("COMPREHENSIVE LEGAL DOCUMENT ANALYSIS")
    print("="*60)

    print("\nDOCUMENT SUMMARY")
    print("-"*40)
    print(f"Method: {summary_result['method'].upper()}")
    print(f"Original Length: {summary_result['original_length']} words")
    print(f"Summary Length: {summary_result['summary_length']} words")
    print(f"Compression Ratio: {summary_result['compression_ratio']:.3f}")
    print(f"\nSummary Text:")
    print(summary_result['summary'])

    print("\nKEY LEGAL CLAUSES")
    print("-"*40)
    doc = nlp(legal_text)
    sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 30]

    clause_scores = {}
    for i, sent in enumerate(sentences):
        score = 0
        legal_keywords = ['agreement', 'contract', 'shall', 'party', 'terminate', 'liability', 'confidential', 'governing', 'jurisdiction']
        for keyword in legal_keywords:
            if keyword in sent.lower():
                score += 1
        clause_scores[sent] = score

    top_clauses = sorted(clause_scores.items(), key=lambda x: x[1], reverse=True)[:5]
    for i, (clause, score) in enumerate(top_clauses, 1):
        print(f"{i}. {clause}")

    print("\nRISK ASSESSMENT")
    print("-"*40)
    if risks:
        risk_categories = {
            'high': ['fraud', 'lawsuit', 'breach'],
            'medium': ['penalty', 'violation', 'noncompliance'],
            'low': ['risk', 'litigation', 'regulatory', 'fine']
        }

        for risk in risks:
            for level, risk_list in risk_categories.items():
                if risk in risk_list:
                    risk_level = level
                    break
            else:
                risk_level = 'low'

            risk_symbols = {'high': '', 'medium': '', 'low': ''}
            print(f"{risk_symbols.get(risk_level, '')} {risk.title()} - {risk_level.upper()} priority")
    else:
        print("No significant legal risks detected")

    print("\nENTITY RECOGNITION")
    print("-"*40)
    if summary_result['entities']:
        for entity_type, entity_list in summary_result['entities'].items():
            if entity_list:
                print(f"{entity_type}: {', '.join(entity_list[:5])}")
    else:
        print("No specific legal entities identified")

    if risks:
        risk_levels = {'high': 0, 'medium': 0, 'low': 0}
        for risk in risks:
            for level, risk_list in {
                'high': ['fraud', 'lawsuit', 'breach'],
                'medium': ['penalty', 'violation', 'noncompliance'],
                'low': ['risk', 'litigation', 'regulatory', 'fine']
            }.items():
                if risk in risk_list:
                    risk_levels[level] += 1
                    break

        print("\nRisk Level Distribution:")
        for level, count in risk_levels.items():
            if count > 0:
                bar_length = count * 5
                bar = "" * bar_length
                symbols = {'high': '', 'medium': '', 'low': ''}
                print(f"{symbols[level]} {level.upper().ljust(6)} │ {bar} {count}")

    print("\nCOMPLIANCE CHECK")
    print("-"*40)
    if compliance_issues:
        for issue in compliance_issues:
            print(f"{issue}")
    else:
        print("Document appears compliant with standard legal requirements")

    print("\nGENERATE COMPREHENSIVE LEGAL REPORT")
    print("-"*40)

    report_content = f"""
LEGAL DOCUMENT ANALYSIS REPORT
{'='*50}

DOCUMENT INFORMATION
- File: {document_path}
- Analysis Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
- Method Used: {summary_result['method'].upper()}

SUMMARY STATISTICS
- Original Length: {summary_result['original_length']} words
- Summary Length: {summary_result['summary_length']} words
- Compression Ratio: {summary_result['compression_ratio']:.3f}

EXECUTIVE SUMMARY
{summary_result['summary']}

KEY LEGAL CLAUSES
{chr(10).join([f'{i}. {clause}' for i, (clause, score) in enumerate(top_clauses, 1)])}

RISK ASSESSMENT
Detected Risks: {len(risks) if risks else 0}
{chr(10).join([f'- {risk}' for risk in risks]) if risks else '- No significant risks detected'}

ENTITY RECOGNITION
{chr(10).join([f'{entity_type}: {", ".join(entity_list)}' for entity_type, entity_list in summary_result['entities'].items() if entity_list])}

COMPLIANCE ANALYSIS
{chr(10).join([f'- {issue}' for issue in compliance_issues]) if compliance_issues else '- Document appears compliant'}

{'='*50}
Report generated by Legal Document Analysis System
"""

    report_filename = f"Legal_Analysis_Report_{Path(document_path).stem}.txt"
    try:
        with open(report_filename, 'w', encoding='utf-8') as f:
            f.write(report_content)
        print(f"Comprehensive report saved to: {report_filename}")
    except Exception as e:
        print(f"Error saving report: {e}")

    print("\n" + "="*60)
    print("ANALYSIS COMPLETE")
    print("="*60)
else:
    print("No document content to analyze")

In [ ]:
def process_and_save_document(file_path, output_file="legal_analysis_output.txt"):
    import os
    from pathlib import Path
    from datetime import datetime
    
    print(f"Processing document: {file_path}")
    
    # Initialize components
    preprocessor = LegalDocumentPreprocessor()
    summarizer = HybridLegalSummarizer()
    reader = DocumentReader()
    
    try:
        # Read document
        doc_data = reader.read_document(file_path)
        
        if not doc_data['success'] or not doc_data['content'].strip():
            print(f"Error: {doc_data.get('error', 'No content found')}")
            return None
        
        legal_text = doc_data['content']
        print(f"Document loaded successfully. Length: {len(legal_text.split())} words")
        
        # Generate summary
        summary_result = summarizer.get_summary_with_metadata(legal_text, method='hybrid')
        
        # Risk assessment
        risk_words = ["fraud", "penalty", "violation", "risk", "lawsuit", "breach", 
                     "noncompliance", "litigation", "regulatory", "fine", "termination",
                     "liability", "confidential", "proprietary"]
        text_lower = legal_text.lower()
        risks = [word for word in risk_words if word in text_lower]
        
        # Compliance analysis
        compliance_issues = []
        if "governing law" not in text_lower and "jurisdiction" not in text_lower:
            compliance_issues.append("Missing governing law or jurisdiction clause")
        if "termination" not in text_lower:
            compliance_issues.append("No termination clause found")
        if "confidential" not in text_lower and "proprietary" not in text_lower:
            compliance_issues.append("No confidentiality clause")
        if "liability" not in text_lower:
            compliance_issues.append("Liability terms not clearly defined")
        
        # Extract key legal clauses
        doc = nlp(legal_text)
        sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 30]
        
        clause_scores = {}
        legal_keywords = ['agreement', 'contract', 'shall', 'party', 'terminate', 'liability', 'confidential', 'governing', 'jurisdiction']
        for i, sent in enumerate(sentences):
            score = sum(1 for keyword in legal_keywords if keyword in sent.lower())
            clause_scores[sent] = score
        
        top_clauses = sorted(clause_scores.items(), key=lambda x: x[1], reverse=True)[:5]
        
        # Method comparison
        methods_comparison = {}
        for method in ['extractive', 'abstractive', 'hybrid']:
            result = summarizer.get_summary_with_metadata(legal_text, method)
            methods_comparison[method] = {
                'summary': result['summary'],
                'compression': result['compression_ratio']
            }
        
        # Generate comprehensive output
        output_content = f"""
LEGAL DOCUMENT ANALYSIS REPORT
{'='*80}

DOCUMENT INFORMATION
- File: {file_path}
- Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
- File Type: {doc_data.get('file_type', 'unknown')}
- Word Count: {len(legal_text.split())}

SUMMARY STATISTICS
- Original Length: {summary_result['original_length']} words
- Summary Length: {summary_result['summary_length']} words
- Compression Ratio: {summary_result['compression_ratio']:.3f}
- Reduction: {(1-summary_result['compression_ratio'])*100:.1f}%

EXECUTIVE SUMMARY (HYBRID METHOD)
{summary_result['summary']}

METHOD COMPARISON
{'-'*50}

EXTRACTIVE SUMMARY
- Compression Ratio: {methods_comparison['extractive']['compression']:.3f}
- Summary: {methods_comparison['extractive']['summary']}

ABSTRACTIVE SUMMARY  
- Compression Ratio: {methods_comparison['abstractive']['compression']:.3f}
- Summary: {methods_comparison['abstractive']['summary']}

HYBRID SUMMARY
- Compression Ratio: {methods_comparison['hybrid']['compression']:.3f}
- Summary: {methods_comparison['hybrid']['summary']}

KEY LEGAL CLAUSES (Top 5)
{'-'*50}
{chr(10).join([f'{i}. {clause}' for i, (clause, score) in enumerate(top_clauses, 1)])}

RISK ASSESSMENT
{'-'*50}
- Total Risks Detected: {len(risks)}
{chr(10).join([f'  - {risk.title()}' for risk in risks]) if risks else '  - No significant risks detected'}

RISK CATEGORIES
{chr(10).join([f'  - {category.upper()}: {", ".join([r for r in risks if r in category_risks])}' 
               for category, category_risks in {
                   'HIGH': ['fraud', 'lawsuit', 'breach'],
                   'MEDIUM': ['penalty', 'violation', 'noncompliance', 'termination', 'liability'],
                   'LOW': ['risk', 'litigation', 'regulatory', 'fine', 'confidential', 'proprietary']
               }.items() if any(r in risks for r in category_risks)]) if risks else '  - No risks categorized'}

COMPLIANCE ANALYSIS
{'-'*50}
{chr(10).join([f'  - {issue}' for issue in compliance_issues]) if compliance_issues else '  - Document appears compliant with standard legal requirements'}

ENTITY RECOGNITION
{'-'*50}
{chr(10).join([f'  - {entity_type}: {", ".join(entity_list[:5])}' 
               for entity_type, entity_list in summary_result['entities'].items() 
               if entity_list]) if summary_result['entities'] else '  - No entities identified'}

DETAILED ANALYSIS
{'-'*50}

DOCUMENT STATISTICS
- Processing Method: Hybrid (Extractive + Abstractive)
- Legal Entities Found: {sum(len(entities) for entities in summary_result['entities'].values())}
- Risk Factors Identified: {len(risks)}
- Compliance Issues: {len(compliance_issues)}
- Key Clauses Extracted: {len(top_clauses)}

QUALITY METRICS
- Summary Coherence: High (Hybrid method)
- Information Retention: {summary_result['compression_ratio']:.1%}
- Risk Coverage: {len(risks)/len(risk_words)*100:.1f}% of risk categories checked
- Compliance Score: {max(0, 100 - len(compliance_issues)*25)}/100

RECOMMENDATIONS
{'-'*50}
{chr(10).join([f'  - {recommendation}' for recommendation in [
    'Review identified risk factors with legal counsel',
    'Address missing compliance clauses if applicable',
    'Consider adding specific termination conditions',
    'Verify confidentiality provisions are adequate',
    'Update governing law and jurisdiction clauses'
    ][:len(compliance_issues)+2]])}

{'='*80}
Report generated by Legal Document Analysis System
Processing completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""
        
        # Save to file
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(output_content)
        
        print(f"Analysis completed successfully!")
        print(f"Comprehensive report saved to: {output_file}")
        print(f"Original document: {summary_result['original_length']} words")
        print(f"Summary length: {summary_result['summary_length']} words")
        print(f"Compression ratio: {summary_result['compression_ratio']:.3f}")
        print(f"Risks detected: {len(risks)}")
        print(f"Compliance issues: {len(compliance_issues)}")
        
        return {
            'output_file': output_file,
            'summary_result': summary_result,
            'risks': risks,
            'compliance_issues': compliance_issues,
            'methods_comparison': methods_comparison
        }
        
    except Exception as e:
        print(f"Error processing document: {e}")
        return None

# USAGE EXAMPLES
print("Legal Document Analysis System")
print("="*50)
print("\nSimple Usage:")
print("result = process_and_save_document('/path/to/document.pdf')")
print("\nCustom Output File:")
print("result = process_and_save_document('/path/doc.pdf', 'my_report.txt')")
print("\nMultiple Documents:")
print("for file in ['doc1.pdf', 'doc2.txt']:")
print("    process_and_save_document(file, f'analysis_{Path(file).stem}.txt')")

# Example usage (uncomment to run):
# result = process_and_save_document('/content/bankj.PDF', 'bankj_analysis.txt')

In [ ]:
# evaluation and confusion matrix 